In [ ]:
%run ../setup_env.py

# Entity-Anchored Retrieval for FEVER

**Extension of notebook 03 — recall-only, no re-classification.**

### Motivation

Two findings motivated this experiment:

1. Lewis et al. (2021) Table 6 shows BM25 outperforms dense retrieval on FEVER — FEVER claims are entity-centric, and token-level matching helps.
2. Our baseline recall (notebook 03) shows REFUTES recall is 7–8 pp below SUPPORTS at every k. The likely cause: DPR matches claim tokens, so for a REFUTES claim like *"The Eiffel Tower is in Berlin"* it retrieves Berlin passages rather than Eiffel Tower passages.

### Hypothesis

Explicitly extracting named entities from the claim, retrieving all passages whose source article matches any entity, then merging with dense results, recovers gold evidence that DPR misses — especially for REFUTES.

### Scope

- 200 validation claims (100 SUPPORTS, 100 REFUTES)
- Measure recall@1/5/10, baseline vs merged, split by label
- No re-classification; recall is the target metric
- Lightweight entity anchoring — not full GraphRAG; the FAISS index is our node set

In [ ]:
import os
import sys
import json
import faiss
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import transformers.utils.import_utils as import_utils
import transformers.utils as tu

# faiss patch (same as other notebooks)
if hasattr(import_utils.is_faiss_available, "cache_clear"):
    import_utils.is_faiss_available.cache_clear()
import_utils._faiss_available = True
import_utils.is_faiss_available = lambda: True
tu.is_faiss_available = lambda: True

# paths
REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), "../.."))
FEVER_DIR   = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR    = os.path.join(FEVER_DIR, "data")
RESULTS_DIR = os.path.join(FEVER_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Fever dir:   {FEVER_DIR}")
print(f"Data dir:    {DATA_DIR}")
print(f"Results dir: {RESULTS_DIR}")
print(f"FAISS patch: {import_utils.is_faiss_available()}")

## Cell 1 — Load passages, FAISS index, FEVER dataset, DPR encoder

In [ ]:
from datasets import load_dataset
from transformers import DPRQuestionEncoder, DPRQuestionEncoderTokenizerFast

# --- passages ---
print("Loading passages...")
passages = []
with open(os.path.join(DATA_DIR, "fever_passages.jsonl")) as f:
    for line in f:
        passages.append(json.loads(line))
print(f"Passages loaded: {len(passages):,}")

# --- FAISS index ---
print("Loading FAISS index...")
index = faiss.read_index(os.path.join(DATA_DIR, "fever_faiss.index"))
print(f"Index vectors:   {index.ntotal:,}")

# --- FEVER dataset ---
print("Loading FEVER dataset...")
dataset = load_dataset("copenlu/fever_gold_evidence")
print(f"Validation examples: {len(dataset['validation']):,}")

# --- DPR question encoder ---
print("Loading DPR question encoder...")
q_tokenizer = DPRQuestionEncoderTokenizerFast.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
)
q_encoder = DPRQuestionEncoder.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
).to("cuda")
q_encoder.eval()
print(f"DPR encoder on: {next(q_encoder.parameters()).device}")

# --- shared utilities ---
def clean_fever_title(title):
    title = title.replace("-LRB-", "(").replace("-RRB-", ")")
    title = title.replace("-LSB-", "[").replace("-RSB-", "]")
    title = title.replace("-LCB-", "{").replace("-RCB-", "}")
    title = title.replace("_", " ")
    return title.strip()

def search_index(claim, index, passages, q_encoder, q_tokenizer, n_docs=10):
    encoded = q_tokenizer(claim, return_tensors="pt", truncation=True, max_length=300)
    with torch.no_grad():
        query_vec = q_encoder(
            input_ids=encoded["input_ids"].to("cuda"),
            attention_mask=encoded["attention_mask"].to("cuda")
        ).pooler_output.cpu().numpy()
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec.astype("float32"), n_docs)
    return [
        {"rank": i + 1, "score": float(s), "title": passages[idx]["title"],
         "text": passages[idx]["text"], "idx": int(idx)}
        for i, (s, idx) in enumerate(zip(scores[0], indices[0]))
    ]

print("\nAll components loaded.")

## Cell 2 — Build article title lookup

Map every normalised article title in our 574K-passage index to the passage indices that
belong to it. This is the "graph" — claim entities map to article titles, which map to
passages already in the FAISS index. No new index is needed.

In [ ]:
print("Building article-title → passage-indices lookup...")
title_to_passage_idxs = {}
for i, p in enumerate(passages):
    key = clean_fever_title(p["title"]).lower()
    if key not in title_to_passage_idxs:
        title_to_passage_idxs[key] = []
    title_to_passage_idxs[key].append(i)

indexed_titles = set(title_to_passage_idxs.keys())
print(f"Unique indexed article titles: {len(indexed_titles):,}")
print(f"Sample titles: {list(indexed_titles)[:5]}")

## Cell 3 — Evaluation sample (200 claims, stratified)

In [ ]:
rng = np.random.default_rng(42)

supports_pool = [
    ex for ex in dataset["validation"]
    if ex["label"] == "SUPPORTS" and ex["evidence"] and ex["evidence"][0][0]
]
refutes_pool = [
    ex for ex in dataset["validation"]
    if ex["label"] == "REFUTES" and ex["evidence"] and ex["evidence"][0][0]
]

supports_sample = [supports_pool[i] for i in rng.choice(len(supports_pool), 100, replace=False)]
refutes_sample  = [refutes_pool[i]  for i in rng.choice(len(refutes_pool),  100, replace=False)]
sample = supports_sample + refutes_sample

print(f"Evaluation sample: {len(sample)} claims")
print(f"  SUPPORTS: {len(supports_sample)}")
print(f"  REFUTES:  {len(refutes_sample)}")

def gold_articles(ex):
    return {clean_fever_title(ev[0]).lower() for ev in ex["evidence"] if ev[0]}

## Cell 4 — spaCy NER + entity→article coverage check

If coverage < 30%, entity anchoring can't work on this index — that itself is a finding.

In [ ]:
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    print("spaCy en_core_web_sm loaded")
except OSError:
    import subprocess
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
    import spacy
    nlp = spacy.load("en_core_web_sm")
    print("spaCy en_core_web_sm downloaded and loaded")

def extract_entity_titles(claim, indexed_titles, max_entities=5):
    """Extract named entities from claim, match to indexed article titles."""
    doc = nlp(claim)
    entities = [ent.text for ent in doc.ents][:max_entities]
    matched = []
    for ent in entities:
        norm = ent.lower()
        if norm in indexed_titles:
            matched.append(norm)
            continue
        # partial: entity text is a substring of an indexed title
        for t in indexed_titles:
            if norm in t:
                matched.append(t)
                break
    return list(dict.fromkeys(matched))  # dedup, preserve order

# --- coverage check ---
n_covered = sum(
    1 for ex in sample
    if extract_entity_titles(ex["claim"], indexed_titles)
)
coverage = n_covered / len(sample)
print(f"\nEntity→article coverage: {coverage:.1%} ({n_covered}/{len(sample)} claims)")

if coverage < 0.30:
    print(
        f"\n[FINDING] Coverage is only {coverage:.1%} — most claim entities do not match"
        " any indexed article title. Entity anchoring cannot improve recall on this index."
        " This is itself a result worth reporting."
    )
else:
    print("Coverage sufficient — proceeding with retrieval experiment.")

## Cell 5 — Baseline dense recall

Reproduce notebook 03 recall on this 200-claim subsample to confirm the harness.
Expected: ~65% top-1 overall, with REFUTES ~7 pp below SUPPORTS.

In [ ]:
DENSE_K = 10

def recall_at_k(titles, gold, k):
    top_k = [t.lower() for t in titles[:k]]
    return any(g in t for g in gold for t in top_k)

baseline = {"SUPPORTS": [], "REFUTES": []}

print(f"Running baseline dense retrieval (top-{DENSE_K}) on {len(sample)} claims...")
for i, ex in enumerate(sample):
    if i % 50 == 0:
        print(f"  {i}/{len(sample)}")
    results = search_index(ex["claim"], index, passages, q_encoder, q_tokenizer, n_docs=DENSE_K)
    titles = [r["title"] for r in results]
    gold   = gold_articles(ex)
    baseline[ex["label"]].append({k: recall_at_k(titles, gold, k) for k in (1, 5, 10)})

print("Done.\n")

def summarise_recall(hits_by_label):
    all_hits = hits_by_label["SUPPORTS"] + hits_by_label["REFUTES"]
    rows = {}
    for label in ("SUPPORTS", "REFUTES", "Overall"):
        hits = hits_by_label.get(label, all_hits)
        rows[label] = {f"top-{k}": f"{np.mean([h[k] for h in hits]):.1%}" for k in (1, 5, 10)}
    return pd.DataFrame(rows).T

# patch Overall in
baseline_all = baseline["SUPPORTS"] + baseline["REFUTES"]
df_baseline = summarise_recall({**baseline, "Overall": baseline_all})
print("Baseline dense recall:")
print(df_baseline.to_string())
print("\n(Notebook 03 reference: Overall top-1 65.4%, SUPPORTS top-1 69.2%, REFUTES top-1 61.5%)")

## Cell 6 — Entity-anchored retrieval + merge

In [ ]:
DENSE_K_MERGE = 5  # dense top-5 for the merged condition

merged = {"SUPPORTS": [], "REFUTES": []}
entity_coverage_per_claim = []

print(f"Running entity-anchored + merged retrieval on {len(sample)} claims...")
for i, ex in enumerate(sample):
    if i % 50 == 0:
        print(f"  {i}/{len(sample)}")

    # dense top-5
    dense_results = search_index(
        ex["claim"], index, passages, q_encoder, q_tokenizer, n_docs=DENSE_K_MERGE
    )
    dense_titles = [r["title"] for r in dense_results]

    # entity-anchored: all passages whose article matches any extracted entity
    entity_matched = extract_entity_titles(ex["claim"], indexed_titles)
    entity_coverage_per_claim.append(len(entity_matched) > 0)
    entity_titles = []
    for t in entity_matched:
        for idx in title_to_passage_idxs.get(t, []):
            entity_titles.append(passages[idx]["title"])

    # merge: dense first, then entity-anchored (dedup by lowercased title)
    seen = set()
    merged_titles = []
    for t in dense_titles + entity_titles:
        norm = t.lower()
        if norm not in seen:
            seen.add(norm)
            merged_titles.append(t)

    gold = gold_articles(ex)
    merged[ex["label"]].append({k: recall_at_k(merged_titles, gold, k) for k in (1, 5, 10)})

print("Done.")
print(f"Claims with ≥1 entity match: {sum(entity_coverage_per_claim)}/{len(sample)} "
      f"({np.mean(entity_coverage_per_claim):.1%})")

## Cell 7 — Results table & chart

In [ ]:
merged_all = merged["SUPPORTS"] + merged["REFUTES"]

# --- summary table ---
rows = []
for label in ("SUPPORTS", "REFUTES", "Overall"):
    b_hits = baseline.get(label, baseline_all)
    m_hits = merged.get(label, merged_all)
    for condition, hits in (("dense", b_hits), ("merged", m_hits)):
        rows.append({
            "label": label,
            "condition": condition,
            "top-1":  np.mean([h[1]  for h in hits]),
            "top-5":  np.mean([h[5]  for h in hits]),
            "top-10": np.mean([h[10] for h in hits]),
        })

df = pd.DataFrame(rows).set_index(["label", "condition"])
df_pct = df.applymap(lambda x: f"{x:.1%}")
print("Recall@k — baseline dense vs entity-anchored merge\n")
print(df_pct.to_string())

# delta: merged minus dense
print("\nDelta (merged − dense):")
for label in ("SUPPORTS", "REFUTES", "Overall"):
    b = df.loc[(label, "dense")]
    m = df.loc[(label, "merged")]
    print(f"  {label:9s}  top-1 {(m['top-1']-b['top-1']):+.1%}  "
          f"top-5 {(m['top-5']-b['top-5']):+.1%}  "
          f"top-10 {(m['top-10']-b['top-10']):+.1%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Entity-Anchored vs Dense Retrieval — Recall@k by Label", fontsize=13)

colors = {"dense": "#3498db", "merged": "#e74c3c"}
labels_plot = ["SUPPORTS", "REFUTES", "Overall"]
k_vals = [1, 5, 10]

for ax, k in zip(axes, k_vals):
    x = np.arange(len(labels_plot))
    w = 0.35
    for j, cond in enumerate(("dense", "merged")):
        vals = [df.loc[(lbl, cond), f"top-{k}"] for lbl in labels_plot]
        bars = ax.bar(x + (j - 0.5) * w, vals, w,
                      label=cond, color=colors[cond], alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.01,
                    f"{v:.0%}", ha="center", va="bottom", fontsize=8)
    ax.set_title(f"Recall@{k}")
    ax.set_xticks(x)
    ax.set_xticklabels(labels_plot, fontsize=9)
    ax.set_ylim(0, 1.10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.legend(fontsize=8)

plt.tight_layout()
out_path = os.path.join(RESULTS_DIR, "fever_kg_recall.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved to {out_path}")

In [ ]:
def hits_to_dict(hits_list):
    return {
        f"top{k}": round(float(np.mean([h[k] for h in hits_list])), 3)
        for k in (1, 5, 10)
    }

out = {
    "n_claims": len(sample),
    "entity_coverage": round(float(np.mean(entity_coverage_per_claim)), 3),
    "dense_k_merge": DENSE_K_MERGE,
    "baseline": {
        "SUPPORTS": hits_to_dict(baseline["SUPPORTS"]),
        "REFUTES":  hits_to_dict(baseline["REFUTES"]),
        "Overall":  hits_to_dict(baseline_all),
    },
    "merged": {
        "SUPPORTS": hits_to_dict(merged["SUPPORTS"]),
        "REFUTES":  hits_to_dict(merged["REFUTES"]),
        "Overall":  hits_to_dict(merged_all),
    },
}

json_path = os.path.join(RESULTS_DIR, "fever_kg_recall.json")
with open(json_path, "w") as f:
    json.dump(out, f, indent=2)
print(f"Results saved to {json_path}")
print(json.dumps(out, indent=2))

## Interpretation

### What we measured

On 200 validation claims (100 SUPPORTS, 100 REFUTES), we compared two retrieval conditions:

- **Dense (baseline):** DPR top-10, same as notebook 03.
- **Merged:** DPR top-5 ∪ all passages from articles whose title matches any spaCy-extracted entity in the claim.

### Core hypothesis test

The key question: does the merged condition lift **REFUTES recall more than SUPPORTS recall**?
If yes, entity anchoring differentially rescues the false-entity problem specific to REFUTES.
If the lift is equal (or SUPPORTS gains more), dense retrieval's failure mode is not label-specific
and the hypothesis does not hold on this index.

### What this is (and is not)

This is **lightweight entity-anchored retrieval** — not full GraphRAG. The "graph" has one hop:
claim entities → Wikipedia article titles → passages already in the FAISS index.
No new index was built; coverage is limited by the 23,733-article index (≈80% of FEVER's
Wikipedia articles).

Re-classification is out of scope. The effect on final label accuracy is left as future work.

### Connection to the paper

Lewis et al. (2021) Table 6 shows BM25 beats DPR on FEVER — this experiment tests the same
phenomenon from a different angle: can explicit entity matching substitute for BM25's token-level
recall advantage, without rebuilding the retriever?